In [1]:
# ============================================================
# IMPORTS
# ============================================================

import pandas as pd
import warnings
import sys
import os

warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath(".."))

from functions.tools import configure_notebook_display, load_raw_datasets

configure_notebook_display()

In [2]:
# ============================================================
# LOADING DATASETS
# ============================================================

df_meta, df_readings = load_raw_datasets()

In [3]:
# ============================================================
# PARCEL COVERAGE CHECK
# ============================================================

meta_parcels = set(df_meta["parcel_id"])
reading_parcels = set(df_readings["parcel_id"])

missing_in_readings = meta_parcels - reading_parcels
missing_in_metadata = reading_parcels - meta_parcels

print("Parcels present in metadata but missing in readings:")
print(missing_in_readings)

print("\nParcels present in readings but missing in metadata:")
print(missing_in_metadata)

Parcels present in metadata but missing in readings:
{'PARCEL_050', 'PARCEL_051', 'PARCEL_052'}

Parcels present in readings but missing in metadata:
{'PARCEL_099', 'PARCEL_098'}


A very important discovery this, some parcels are not in the readings and not in the metadata and vice versa. This could suggest that some parcels were meant for testing only, on early deployment phase or needs parcel not activated inititally.

In [4]:
# ============================================================
# PARCEL-DATE DUPLICATE CHECK
# ============================================================

parcel_date_duplicates = df_readings.duplicated(
    subset=["parcel_id", "date"]).sum()

print(f"Duplicate parcel-date rows: {parcel_date_duplicates}")

Duplicate parcel-date rows: 8


In earlier duplicate checks, we found out no entries in readings is duplicated, here we are checking parcel + date to verify if some parcels take reading more than once per day. As only 8 times it is suggested, we can conclude it is nothing out of ordinary.

In [5]:
# ============================================================
# READING COUNT PER PARCEL
# ============================================================

parcel_reading_counts = (
    df_readings["parcel_id"]
    .value_counts()
    .reset_index())

parcel_reading_counts.columns = ["parcel_id", "reading_count"]

display(parcel_reading_counts)

,parcel_id,reading_count
0,PARCEL_007,142
1,PARCEL_023,141
2,PARCEL_010,140
3,PARCEL_025,140
4,PARCEL_016,140
5,PARCEL_002,139
6,PARCEL_012,139
7,PARCEL_013,139
8,PARCEL_009,139
9,PARCEL_020,138


`PARCEL_099` & `PARCEL_098`, these parcels which were not present in the metadata are only showing 20 readings each. 

While rest of the Parcels have normal count.

In [6]:
# ============================================================
# DATE PARSING FOR TEMPORAL AUDIT
# ============================================================

df_readings["parsed_date"] = pd.to_datetime(
    df_readings["date"],
    format="mixed",
    dayfirst=True)

parcel_date_ranges = (
    df_readings
    .groupby("parcel_id")["parsed_date"]
    .agg(["min", "max", "count"])
    .reset_index())

parcel_date_ranges["date_span_days"] = (
    parcel_date_ranges["max"] - parcel_date_ranges["min"]).dt.days

parcel_date_ranges.columns = [
    "parcel_id",
    "first_recorded_date",
    "last_recorded_date",
    "total_records",
    "date_span_days"
]

display(parcel_date_ranges)

,parcel_id,first_recorded_date,last_recorded_date,total_records,date_span_days
0,PARCEL_001,2026-01-01,2026-12-05,136,338
1,PARCEL_002,2026-01-01,2026-12-05,139,338
2,PARCEL_003,2026-01-01,2026-12-05,136,338
3,PARCEL_004,2026-01-01,2026-12-04,134,337
4,PARCEL_005,2026-01-01,2026-12-05,132,338
5,PARCEL_006,2026-01-01,2026-12-04,136,337
6,PARCEL_007,2026-01-01,2026-12-04,142,337
7,PARCEL_008,2026-01-01,2026-12-05,134,338
8,PARCEL_009,2026-01-01,2026-12-05,139,338
9,PARCEL_010,2026-01-01,2026-12-05,140,338


Here we applied for first reading and last reading date of each parcels, and got the difference to see any of the parcels stopped registereing. The last date of every parcel is december where as the two parcel 099 and 098 ends in october and november respectively. 

This suggests that 098 and 099 may be pulled from field earlier. 

In [7]:
# ============================================================
# INVALID NDVI COUNT BY PARCEL
# ============================================================

invalid_ndvi_by_parcel = (
    df_readings[
        (df_readings["ndvi_value"] < -1) |
        (df_readings["ndvi_value"] > 1)
    ]
    ["parcel_id"]
    .value_counts()
    .reset_index())

invalid_ndvi_by_parcel.columns = ["parcel_id", "invalid_ndvi_count"]

display(invalid_ndvi_by_parcel)

,parcel_id,invalid_ndvi_count
0,PARCEL_017,9
1,PARCEL_024,8
2,PARCEL_014,7
3,PARCEL_007,7
4,PARCEL_012,6
5,PARCEL_011,6
6,PARCEL_009,6
7,PARCEL_001,5
8,PARCEL_019,5
9,PARCEL_025,5


Fairly simple analysis here to see how many out of range, invalid NDVI were recorded by each parcels and those Invalid NDVI have a normal distribution.

In [8]:
# ============================================================
# INVALID NDVI COUNT BY MILL
# ============================================================

merged_df = df_readings.merge(
    df_meta,
    on="parcel_id",
    how="left")

invalid_ndvi_by_mill = (
    merged_df[
        (merged_df["ndvi_value"] < -1) |
        (merged_df["ndvi_value"] > 1)]
    ["mill_id"]
    .value_counts(dropna=False)
    .reset_index())

invalid_ndvi_by_mill.columns = ["mill_id", "invalid_ndvi_count"]

display(invalid_ndvi_by_mill)

,mill_id,invalid_ndvi_count
0,MILL_EAST,36
1,MILL_NORTH,36
2,MILL_WEST,18
3,MILL_SOUTH,14


In [9]:
# ============================================================
# MISSING SENSOR STATUS BY PARCEL
# ============================================================

missing_status_by_parcel = (
    df_readings[
        df_readings["sensor_status"].isna()]
    ["parcel_id"]
    .value_counts()
    .reset_index())

missing_status_by_parcel.columns = ["parcel_id", "missing_status_count"]

display(missing_status_by_parcel)

,parcel_id,missing_status_count
0,PARCEL_006,10
1,PARCEL_025,9
2,PARCEL_017,8
3,PARCEL_012,7
4,PARCEL_019,7
5,PARCEL_013,7
6,PARCEL_005,7
7,PARCEL_009,7
8,PARCEL_016,7
9,PARCEL_020,7


Just like invalid NDVI for each Parcels, missing values per parcel is also stable with nothing out of ordinary.

In [10]:
# Updated tracker from notebook 01
issue_tracker = pd.DataFrame({

    "column": [
        "crop_type",
        "sowing_date",
        "date",
        "ndvi_value",
        "sensor_status",
        "parcel_id",
    ],

    "issue_identified": [

        "Imbalanced crop distribution",
        "Stored as string instead of datetime",
        "Multiple date formats within the same column",
        "NDVI values outside valid biological range [-1, 1]",
        "Dirty categorical labels and missing statuses",
        "Parcel inconsistency between metadata and readings",

    ],

    "prevalence": [

        "Sugarcane dominates by ~ 70% of all parcels",
        "Entire column",
        "3 types of formats detected",
        "104 invalid records detected",
        "Whitespace, casing issues and 137 missing values detected",
        "3 parcels absent in readings and 2 absent in metadata",
    ],

    "severity": [

        "Low",
        "Medium",
        "High",
        "High",
        "Medium",
        "High",
    ]
})

display(issue_tracker)

,column,issue_identified,prevalence,severity
0,crop_type,Imbalanced crop distribution,Sugarcane dominates by ~ 70% of all parcels,Low
1,sowing_date,Stored as string instead of datetime,Entire column,Medium
2,date,Multiple date formats within the same column,3 types of formats detected,High
3,ndvi_value,"NDVI values outside valid biological range [-1, 1]",104 invalid records detected,High
4,sensor_status,Dirty categorical labels and missing statuses,"Whitespace, casing issues and 137 missing values detected",Medium
5,parcel_id,Parcel inconsistency between metadata and readings,3 parcels absent in readings and 2 absent in metadata,High
